# SARA · Audio

For first setup, follow the [Linux installation guide](../README.md#install-on-linux) and run `make notebooks` to install the kernel.

1. Run `make ui` in the SARA folder and open the printed HTTPS URL on the phone.
2. Select **Audio** on the phone and this project's **.venv** kernel in VS Code.
3. Run the cells below. Start the recording cell, then hold **Hold to capture**, speak, and release.

The notebook reads live audio through `/api/audio.wav` and keeps its own captured samples.
The app's normal voice workflow also processes the phone recording on release.

## Connect

Run this notebook on the gateway computer. `GATEWAY` is the data source address.
Monitoring uses the admin password, which defaults to `admin`.

In [ ]:
import json
import time
from getpass import getpass
from http.cookiejar import CookieJar
from pathlib import Path
from urllib.request import HTTPCookieProcessor, Request, build_opener

PROJECT = next(path for path in (Path.cwd(), *Path.cwd().parents)
               if (path / "gateway/server.py").exists())
GATEWAY = "http://localhost:8080"
connection = build_opener(HTTPCookieProcessor(CookieJar()))
password = getpass("Admin password (Enter for admin): ") or "admin"
login = Request(GATEWAY + "/api/admin/login",
                data=json.dumps({"password": password}).encode(),
                headers={"Content-Type": "application/json"})
with connection.open(login, timeout=10) as response:
    print("Connected to SARA")
password = None


def get(path):
    with connection.open(GATEWAY + path, timeout=10) as response:
        return response.read(), response.headers

## Collect and listen

Run this cell, then hold the phone's capture button and speak. Release to finish.
The cell waits for up to 20 seconds. `audio` is a NumPy float32 array of mono samples at
`SAMPLE_RATE = 16000`, ready for an audio model. It stays in memory after the phone disconnects.

`read_audio(after)` reads new samples and returns the next sequence number, preventing duplicates.
The gateway's live buffer holds five seconds, so this example reads it frequently.

In [ ]:
from io import BytesIO
import wave
import numpy as np
from IPython.display import Audio, display

SAMPLE_RATE = 16000


def read_audio(after=0):
    data, headers = get(f"/api/audio.wav?after={after}")
    if not data:
        return np.empty(0, dtype=np.float32), after
    with wave.open(BytesIO(data), "rb") as recording:
        pcm = recording.readframes(recording.getnframes())
    samples = np.frombuffer(pcm, dtype="<i2").astype(np.float32) / 32768
    return samples, int(headers["X-Audio-Sequence"])


def record_audio(wait_seconds=20):
    after = json.loads(get("/api/status")[0])["audio_sequence"]
    chunks = []
    deadline = time.monotonic() + wait_seconds
    print("Hold the capture button, speak, then release to finish.")
    while time.monotonic() < deadline:
        samples, after = read_audio(after)
        if samples.size:
            chunks.append(samples)
        if chunks and json.loads(get("/api/status")[0])["state"] == "idle":
            break
        time.sleep(0.1)
    return np.concatenate(chunks) if chunks else np.empty(0, dtype=np.float32)


audio = record_audio()
if audio.size:
    print(f"Captured {audio.size / SAMPLE_RATE:.1f} seconds at {SAMPLE_RATE} Hz")
    display(Audio(audio, rate=SAMPLE_RATE))
else:
    print("No audio received. Select Audio on the phone and run this cell again.")

## Try speech recognition

This example uses faster-whisper's `tiny` model on the CPU. The weights are reused from
`.local/models` and downloaded on first use if needed.

Edit `transcribe(audio)` to try another speech model, sound classifier, or detection logic.
Rerun the recording cell for new input and reuse `audio` for repeated offline tests.
[Model documentation](https://github.com/SYSTRAN/faster-whisper).

In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel("tiny", device="cpu", compute_type="int8", cpu_threads=2,
                     download_root=str(PROJECT / ".local/models"))


def transcribe(audio):
    segments, _ = model.transcribe(audio, beam_size=1, vad_filter=True,
                                  condition_on_previous_text=False)
    return " ".join(segment.text.strip() for segment in segments)


if audio.size:
    print(transcribe(audio))